In [27]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["praktikum_bigdata"]

# Memilih (atau otomatis membuat) collection
koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum-bigdata', 'praktikum_bigdata']


In [28]:
!start-dfs.sh
!start-yarn.sh

Starting namenodes on [localhost]
Starting datanodes
Starting secondary namenodes [fais]
Starting resourcemanager
Starting nodemanagers


---
## Latihan Mandiri

Pastikan `client`, `db`, dan `koleksi_produk` dari Sub-bab 9.4 masih aktif sebelum mengerjakan latihan berikut (jalankan ulang cellnya jika kernel sudah di-restart).

In [29]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["praktikum_bigdata"]

# Memilih (atau otomatis membuat) collection
koleksi_produk = db["produk"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum-bigdata', 'praktikum_bigdata']


In [30]:
# Menghapus data lama jika ada (agar percobaan kita mulai dari kondisi bersih)
koleksi_produk.delete_many({})

# insert_one() — menyisipkan satu dokumen
koleksi_produk.insert_one({
    "nama_produk": "Kopi Susu",
    "kategori": "Minuman",
    "harga": 18000,
    "stok": 120,
    "tags": ["best-seller", "dingin"]
})

# insert_many() — menyisipkan banyak dokumen sekaligus
koleksi_produk.insert_many([
    {"nama_produk": "Roti Bakar", "kategori": "Makanan", "harga": 15000, "stok": 45, "tags": ["hangat"]},
    {"nama_produk": "Es Teh", "kategori": "Minuman", "harga": 8000, "stok": 200, "tags": ["dingin", "murah"]},
    {"nama_produk": "Kentang Goreng", "kategori": "Makanan", "harga": 20000, "stok": 60, "tags": ["gurih"]},
    {"nama_produk": "Cappuccino", "kategori": "Minuman", "harga": 25000, "stok": 80, "tags": ["hangat", "best-seller"]},
])

print("Jumlah dokumen di collection 'produk':", koleksi_produk.count_documents({}))

Jumlah dokumen di collection 'produk': 5


In [31]:
# find_one() — mengambil satu dokumen pertama yang cocok
satu_produk = koleksi_produk.find_one({"kategori": "Minuman"})
print("Satu produk kategori Minuman:", satu_produk)

print()

# find() — mengambil SEMUA dokumen yang cocok (mengembalikan cursor, di-looping atau di-list())
print("Seluruh produk kategori Minuman:")
for p in koleksi_produk.find({"kategori": "Minuman"}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

print()

# Filter dengan operator perbandingan: harga lebih dari 15000
print("Produk dengan harga > 15000:")
for p in koleksi_produk.find({"harga": {"$gt": 15000}}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

Satu produk kategori Minuman: {'_id': ObjectId('6aeafd1d78ab54188b74c850'), 'nama_produk': 'Kopi Susu', 'kategori': 'Minuman', 'harga': 18000, 'stok': 120, 'tags': ['best-seller', 'dingin']}

Seluruh produk kategori Minuman:
  - Kopi Susu (Rp18,000)
  - Es Teh (Rp8,000)
  - Cappuccino (Rp25,000)

Produk dengan harga > 15000:
  - Kopi Susu (Rp18,000)
  - Kentang Goreng (Rp20,000)
  - Cappuccino (Rp25,000)


In [32]:
# update_one() — memperbarui SATU dokumen pertama yang cocok
koleksi_produk.update_one(
    {"nama_produk": "Es Teh"},
    {"$set": {"harga": 9000}}   # menaikkan harga Es Teh
)

# update_many() — memperbarui SEMUA dokumen yang cocok
# Contoh: mengurangi stok seluruh produk kategori Minuman sebanyak 5 (simulasi penjualan)
koleksi_produk.update_many(
    {"kategori": "Minuman"},
    {"$inc": {"stok": -5}}   # $inc mengubah nilai relatif (increment/decrement)
)

print("Setelah update:")
for p in koleksi_produk.find({}, {"_id": 0, "nama_produk": 1, "harga": 1, "stok": 1}):
    print(f"  - {p['nama_produk']}: harga=Rp{p['harga']:,}, stok={p['stok']}")

Setelah update:
  - Kopi Susu: harga=Rp18,000, stok=115
  - Roti Bakar: harga=Rp15,000, stok=45
  - Es Teh: harga=Rp9,000, stok=195
  - Kentang Goreng: harga=Rp20,000, stok=60
  - Cappuccino: harga=Rp25,000, stok=75


In [33]:
# delete_one() — menghapus SATU dokumen pertama yang cocok
koleksi_produk.delete_one({"nama_produk": "Kentang Goreng"})

print("Jumlah dokumen tersisa:", koleksi_produk.count_documents({}))
print("Daftar produk tersisa:", [p["nama_produk"] for p in koleksi_produk.find({}, {"nama_produk": 1, "_id": 0})])

Jumlah dokumen tersisa: 4
Daftar produk tersisa: ['Kopi Susu', 'Roti Bakar', 'Es Teh', 'Cappuccino']


**Soal 1.** Sisipkan (`insert_many`) lima dokumen baru ke collection `produk` dengan kategori bebas (boleh kategori baru selain Makanan/Minuman, misalnya "Snac%)k").

In [34]:
koleksi_produk.insert_many([
    {"nama_produk": "Martabak", "kategori": "Makanan", "harga": 18000, "stok": 20, "tags": ["hangat"]},
    {"nama_produk": "Es Kopi Susu", "kategori": "Minuman", "harga": 1500, "stok": 40, "tags": ["dingin", "murah"]},
    {"nama_produk": "Pizza", "kategori": "Makanan", "harga": 28000, "stok": 70, "tags": ["gurih"]},
    {"nama_produk": "Latte", "kategori": "Minuman", "harga": 16000, "stok": 30, "tags": ["hangat"]},
])
print("Jumlah dokumen di collection 'produk':", koleksi_produk.count_documents({}))

Jumlah dokumen di collection 'produk': 8


**Soal 2.** Gunakan `find()` dengan operator `$lt` (kurang dari) untuk menampilkan seluruh produk dengan `harga` di bawah 20000.

In [35]:
print("Produk dengan harga < 2000:")
for p in koleksi_produk.find({"harga": {"$lt":2000}}):
    print(f" - {p['nama_produk']} (Rp{p['harga']:,})")

Produk dengan harga < 2000:
 - Es Kopi Susu (Rp1,500)


**Soal 3.** Gunakan `update_many()` untuk menambahkan field baru bernama `diskon` bernilai `0` ke **seluruh** dokumen pada collection `produk` (petunjuk: gunakan operator `$set`).

In [36]:
koleksi_produk.update_many(
    {},
    {"$set": {"diskon": 0}}
)

print("setelah update:")
for p in koleksi_produk.find({}):
    print(f" - {p['nama_produk']}: harga=Rp{p['harga']:,}, stok={p['stok']}, diskon={p['diskon']}")

setelah update:
 - Kopi Susu: harga=Rp18,000, stok=115, diskon=0
 - Roti Bakar: harga=Rp15,000, stok=45, diskon=0
 - Es Teh: harga=Rp9,000, stok=195, diskon=0
 - Cappuccino: harga=Rp25,000, stok=75, diskon=0
 - Martabak: harga=Rp18,000, stok=20, diskon=0
 - Es Kopi Susu: harga=Rp1,500, stok=40, diskon=0
 - Pizza: harga=Rp28,000, stok=70, diskon=0
 - Latte: harga=Rp16,000, stok=30, diskon=0


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: berikan **satu contoh skenario nyata** (boleh di luar konteks e-commerce yang sering kita pakai) di mana MongoDB akan menjadi pilihan lebih tepat dibanding tabel relasional biasa, dan jelaskan alasannya berdasarkan konsep *flexible schema*. Tulis jawaban pada markdown cell di bawah ini.

**MongoDB menjadi pilihan yang lebih tepat untuk aplikasi rekam medis elektronik (Electronic Health Record) yang menyimpan data pasien dari berbagai klinik. Setiap pasien dapat memiliki riwayat penyakit, alergi, atau hasil laboratorium yang memiliki struktur berbeda-beda (skema fleksibel) tanpa perlu mengubah struktur tabel utama atau mengisi kolom kosong (null) secara masif. Hal ini membuat penambahan atribut medis baru dapat dilakukan dengan cepat tanpa merusak data lama yang sudah ada.**


---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 10 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas9_[NIM]_[Nama Lengkap].ipynb`**, koneksikan ke MongoDB, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun Katalog (Create)** *(bobot 20%)*

Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

**B. Kueri Bertingkat (Read)** *(bobot 20%)*

Tampilkan hasil dari **tiga** kueri berikut :
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali.

**C. Pembaruan Data (Update)** *(bobot 15%)*

Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

**D. Integrasi ke Spark** *(bobot 25%)*

Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame (ikuti pola Sub-bab 9.6 — perhatikan field yang mungkin kosong/`null` pada sebagian dokumen, seperti `tingkat_kepahitan` yang hanya dimiliki menu Kopi). Tampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

**E. Kesimpulan** *(bobot 20%)*

Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas9_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan MongoDB (`sudo systemctl status mongod`) aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Create | Minimal 12 dokumen tersimpan dengan field sesuai ketentuan kategori | 20% |
| B. Read | Ketiga kueri menghasilkan output yang tepat sesuai instruksi | 20% |
| C. Update | Field `status` berhasil ditambahkan & diperbarui sesuai instruksi | 15% |
| D. Integrasi Spark | Konversi ke Spark DataFrame berhasil; analisis `groupBy` benar | 25% |
| E. Kesimpulan | Refleksi menunjukkan pemahaman konseptual yang tepat tentang tantangan skema | 20% |


In [37]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /user/mahasiswa/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /user/mahasiswa/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

put: Cannot create file/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv._COPYING_. Name node is in safe mode.
put: Cannot create file/user/mahasiswa/tugas7/raw/tugas7_cabang.json._COPYING_. Name node is in safe mode.
put: Cannot create file/user/mahasiswa/tugas7/raw/tugas7_menu.csv._COPYING_. Name node is in safe mode.
Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [38]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB yang berjalan di localhost, port default 27017
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["kopi_nusantara"]

# Memilih (atau otomatis membuat) collection
menu = db["menu"]

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum-bigdata', 'praktikum_bigdata']


In [39]:
menu.insert_many([
    {"nama_menu": "Martabak", "kategori": "Makanan", "harga": 18000, "stok": 20, "pedas_level": 0},
    {"nama_menu": "Es Kopi Susu", "kategori": "Kopi", "harga": 1500, "stok": 40, "tingkat_kepahitan": 4},
    {"nama_menu": "Pizza", "kategori": "Makanan", "harga": 28000, "stok": 70, "pedas_level": 2},
    {"nama_menu": "Latte", "kategori": "Kopi", "harga": 16000, "stok": 30, "tingkat_kepahitan": 2},
    {"nama_menu": "Cireng", "kategori": "Makanan", "harga": 18000, "stok": 20, "pedas_level": 3},
    {"nama_menu": "Kopi Hitam", "kategori": "Kopi", "harga": 1500, "stok": 40, "tingkat_kepahitan": 5},
    {"nama_menu": "Mendoan", "kategori": "Makanan", "harga": 28000, "stok": 70, "pedas_level": 1},
    {"nama_menu": "Latte", "kategori": "Kopi", "harga": 16000, "stok": 30, "tingkat_kepahitan": 2},
    {"nama_menu": "Nasi_Goreng", "kategori": "Makanan", "harga": 18000, "stok": 20, "pedas_level": 1},
    {"nama_menu": "Cappucino", "kategori": "Kopi", "harga": 1500, "stok": 40, "tingkat_kepahitan": 3},
    {"nama_menu": "Caramel", "kategori": "Kopi", "harga": 28000, "stok": 70, "tingkat_kepahitan": 2},
    {"nama_menu": "Latte", "kategori": "Kopi", "harga": 16000, "stok": 30, "tingkat_kepahitan": 1},
])
print("Jumlah dokumen di collection 'menu':", menu.count_documents({}))

Jumlah dokumen di collection 'menu': 24


In [40]:
for p in menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f" -{p['nama_menu']} (Rp{p['harga']:,}) - Tingkat Kepahitan: {p['tingkat_kepahitan']} ")

 -Es Kopi Susu (Rp1,500) - Tingkat Kepahitan: 4 
 -Kopi Hitam (Rp1,500) - Tingkat Kepahitan: 5 
 -Es Kopi Susu (Rp1,500) - Tingkat Kepahitan: 4 
 -Kopi Hitam (Rp1,500) - Tingkat Kepahitan: 5 


In [41]:
for p in menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f" -{p['nama_menu']} (Rp{p['harga']:,})")

 -Martabak (Rp18,000)
 -Pizza (Rp28,000)
 -Latte (Rp16,000)
 -Cireng (Rp18,000)
 -Mendoan (Rp28,000)
 -Latte (Rp16,000)
 -Nasi_Goreng (Rp18,000)
 -Caramel (Rp28,000)
 -Latte (Rp16,000)
 -Martabak (Rp18,000)
 -Pizza (Rp28,000)
 -Latte (Rp16,000)
 -Cireng (Rp18,000)
 -Mendoan (Rp28,000)
 -Latte (Rp16,000)
 -Nasi_Goreng (Rp18,000)
 -Caramel (Rp28,000)
 -Latte (Rp16,000)


In [42]:
for p in menu.find({"pedas_level": {"$exists": False}}):
    print(f" -{p['nama_menu']} (Kategori: {p['kategori']})")

 -Es Kopi Susu (Kategori: Kopi)
 -Latte (Kategori: Kopi)
 -Kopi Hitam (Kategori: Kopi)
 -Latte (Kategori: Kopi)
 -Cappucino (Kategori: Kopi)
 -Caramel (Kategori: Kopi)
 -Latte (Kategori: Kopi)
 -Es Kopi Susu (Kategori: Kopi)
 -Latte (Kategori: Kopi)
 -Kopi Hitam (Kategori: Kopi)
 -Latte (Kategori: Kopi)
 -Cappucino (Kategori: Kopi)
 -Caramel (Kategori: Kopi)
 -Latte (Kategori: Kopi)


In [43]:
menu.update_many(
    {},
    {"$set": {"status": "Tersedia"}}
)

menu.update_one(
    {"nama_menu": "Martabak"},
    {"$set": {"status": "Habis"}}
)

menu.update_one(
    {"nama_menu": "Mendoan"},
    {"$set": {"status": "Habis"}}
)

print("Setelah update:")
for p in menu.find({}):
    print(f" - {p['nama_menu']}: Kategori = {p['kategori']}, Harga = Rp{p['harga']:,}, Status = {p['status']}, ")

Setelah update:
 - Martabak: Kategori = Makanan, Harga = Rp18,000, Status = Habis, 
 - Es Kopi Susu: Kategori = Kopi, Harga = Rp1,500, Status = Tersedia, 
 - Pizza: Kategori = Makanan, Harga = Rp28,000, Status = Tersedia, 
 - Latte: Kategori = Kopi, Harga = Rp16,000, Status = Tersedia, 
 - Cireng: Kategori = Makanan, Harga = Rp18,000, Status = Tersedia, 
 - Kopi Hitam: Kategori = Kopi, Harga = Rp1,500, Status = Tersedia, 
 - Mendoan: Kategori = Makanan, Harga = Rp28,000, Status = Habis, 
 - Latte: Kategori = Kopi, Harga = Rp16,000, Status = Tersedia, 
 - Nasi_Goreng: Kategori = Makanan, Harga = Rp18,000, Status = Tersedia, 
 - Cappucino: Kategori = Kopi, Harga = Rp1,500, Status = Tersedia, 
 - Caramel: Kategori = Kopi, Harga = Rp28,000, Status = Tersedia, 
 - Latte: Kategori = Kopi, Harga = Rp16,000, Status = Tersedia, 
 - Martabak: Kategori = Makanan, Harga = Rp18,000, Status = Tersedia, 
 - Es Kopi Susu: Kategori = Kopi, Harga = Rp1,500, Status = Tersedia, 
 - Pizza: Kategori = Makan

In [44]:
from pyspark.sql import SparkSession

# 1. Inisialisasi SparkSession dengan Driver MongoDB
spark = SparkSession.builder \
    .appName("Tugas9_MongoDB_to_Spark") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
    .config("spark.mongodb.read.connection.uri", "mongodb://127.0.0.1:27017/kopi_nusantara.menu") \
    .config("spark.mongodb.write.connection.uri", "mongodb://127.0.0.1:27017/kopi_nusantara.menu") \
    .getOrCreate()

# 2. Membaca data langsung dari MongoDB ke Spark DataFrame
df_spark = spark.read.format("mongodb").load()

print("=== 1. Hasil printSchema() ===")
df_spark.printSchema()

print("\n=== 2. Rata-rata Harga per Kategori ===")
df_spark.groupBy("kategori").avg("harga").show()


=== 1. Hasil printSchema() ===
root
 |-- _id: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- pedas_level: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- stok: integer (nullable = true)
 |-- tingkat_kepahitan: integer (nullable = true)


=== 2. Rata-rata Harga per Kategori ===
+--------+----------+
|kategori|avg(harga)|
+--------+----------+
| Makanan|   22000.0|
|    Kopi|   11500.0|
+--------+----------+



**Tantangan Konversi Data MongoDB ke Spark DataFrame dan Solusinya:**

Saat melakukan konversi data dari MongoDB ke Apache Spark DataFrame, tantangan utama yang dihadapi terletak pada benturan filosofi struktur data kedua platform tersebut. MongoDB menganut prinsip *schemaless* atau *flexible schema*, di mana setiap dokumen bersifat independen dan dapat memiliki field yang berbeda. Pada kasus katalog "Kopi Nusantara", dokumen kategori "Kopi" memiliki field `tingkat_kepahitan` tetapi tidak memiliki `pedas_level`, dan sebaliknya berlaku untuk kategori "Makanan". Beberapa dokumen juga memiliki field opsional seperti `stok` atau `rekomendasi_barista`. 

Sebaliknya, Apache Spark DataFrame memerlukan struktur skema yang kaku dan seragam berbentuk tabular (*structured matrix*) agar bisa menjalankan pemrosesan paralel secara terdistribusi dengan efisien. Ketika Spark membaca data yang tidak seragam ini langsung dari MongoDB, tantangannya adalah bagaimana memetakan ketidakhadiran field tertentu tanpa merusak integritas tipe data kolom tersebut.

Untuk mengatasi hal tersebut, *Mongo Spark Connector* secara otomatis melakukan pemindaian awal (*sampling/schema inference*) terhadap struktur dokumen di MongoDB. Solusi teknis yang diterapkan oleh sistem saat membentuk DataFrame adalah dengan menyatukan seluruh variasi field yang unik menjadi kolom-kolom baru di Spark. Bagi dokumen yang tidak memiliki field tertentu (misalnya menu makanan yang tidak punya kolom `tingkat_kepahitan`), Spark akan secara otomatis mengisinya dengan nilai kosong atau `null` (bersifat *nullable*). Melalui pendekatan toleransi nilai `null` ini, Spark tetap dapat mempertahankan struktur baris dan kolom yang seragam tanpa menghilangkan variasi data dinamis bawaan dari MongoDB, sehingga fungsi analisis data seperti agregasi `groupBy` tetap dapat dieksekusi dengan baik.